# 20. Performance Evidence and Experiment Design | 性能证据汇总与实验设计

**难度：** Medium | **环境：** CPU-first | **标签：** `evidence`, `baseline`, `experiment design` | **目标人群：** Part 2-4 前置补课者

## 本节导读

本节把分散的时间、容量与正确性记录整理成可复查的性能证据，并建立比较基线与候选方案的共同口径。

完成后，你应能从已有观测提出可验证假设，设计只改变一个条件的对照，并根据重复测量、失败记录和质量门槛决定接受、复测或拒绝候选。

**关键词：** `baseline`, `hypothesis`, `single-variable experiment`, `evidence level`

![性能证据汇总与实验设计](../docs/public/00_Prerequisites/20_performance_evidence_experiment_map.svg)

## 前置阅读
**导语：** 完成 17 的时间观察、18 的显存账本和 19 的异常定位后，进入本页把证据整理成下一步实验决策。
- [17. PyTorch Profiling Basics | PyTorch 性能分析基础](./17_PyTorch_Profiling_Basics.ipynb)
- [18. Memory Profiling and Optimization | 显存分析与优化](./18_Memory_Profiling_and_Optimization.ipynb)
- [19. Debugging and Anomaly Localization | 调试与异常定位](./19_Debugging_and_Anomaly_Localization.ipynb)


## Q1：如何把一次运行的性能信号整理成可比较的记录？

假设你已经从一次运行中得到 step 时间、处理 token 数和资源观察值。两次运行的延迟都是 20 ms，但一次处理 1 个样本、另一次处理 8 个样本，它们的吞吐并不相同。这里先把原始条件和带单位指标整理成记录，不根据单个利用率或等待时间直接判定瓶颈。

| 记录字段 | 它回答的问题 | 输出或观察方向 |
|:---|:---|:---|
| latency | 一次运行花了多久？ | 判断单次响应是否变慢 |
| throughput | 单位时间处理了多少 token？ | 判断整体产出是否下降 |
| GPU 利用率 | GPU 是否持续有工作？ | 偏低时检查输入或同步 |
| CPU 等待时间 | CPU 是否在等待数据或设备？ | 偏高时检查数据管线或同步 |
| workload 条件 | 当前结果在什么条件下产生？ | 保留模型、输入、batch、dtype 和设备 |


In [ ]:
def summarize_step_metrics(step_ms, tokens, accelerator_busy_pct, cpu_wait_ms, workload):
    """保留 workload 与带单位指标，不在记录阶段直接判定瓶颈。"""
    if step_ms <= 0 or tokens < 0:
        raise ValueError('step_ms 必须大于 0，tokens 不能为负数')
    if not 0 <= accelerator_busy_pct <= 100 or cpu_wait_ms < 0:
        raise ValueError('设备忙碌率应在 0 到 100 之间，CPU 等待时间不能为负数')
    required = {'model', 'input_shape', 'batch_size', 'dtype', 'device'}
    if not required <= set(workload):
        raise KeyError(f'workload 缺少字段: {sorted(required - set(workload))}')
    tok_per_s = tokens / (step_ms / 1000.0)
    return {
        'workload': dict(workload),
        'latency_ms': step_ms,
        'throughput_tok_s': round(tok_per_s, 1),
        'accelerator_busy_pct': accelerator_busy_pct,
        'cpu_wait_ms': cpu_wait_ms,
    }


workload = {
    'model': 'TinyLM', 'input_shape': (4, 128), 'batch_size': 4,
    'dtype': 'fp32', 'device': 'cpu',
}
examples = [
    dict(step_ms=180, tokens=360, accelerator_busy_pct=42, cpu_wait_ms=55, workload=workload),
    dict(step_ms=260, tokens=520, accelerator_busy_pct=86, cpu_wait_ms=8, workload=workload),
]
summaries = [summarize_step_metrics(**kwargs) for kwargs in examples]
for result in summaries:
    print(result)


## Q1 验证：带单位指标和 workload 条件是否完整保留？

In [ ]:
assert summaries[0]['throughput_tok_s'] == 2000.0
assert summaries[0]['workload'] == workload
assert all('accelerator_busy_pct' in item and 'cpu_wait_ms' in item for item in summaries)
print('✅ workload 与带单位性能记录通过')

## Q2：如何从阶段耗时中选出最值得展开的热点？

假设你已经从 17 节得到一组阶段事件。每条记录包含 `name`、`total_ms` 和 `calls`，排序后再派生 `share_pct`。热点是对总耗时贡献最大的阶段；它用于选择下一项检查，还不能单独证明根因。

| 阶段 | 观察对象 | 下一项验证 |
|:---|:---|:---|
| dataloader | 数据准备和搬运 | 检查输入管线与同步 |
| forward | 前向算子与激活 | 展开算子耗时和显存 |
| backward | 反向计算与梯度 | 检查梯度计算和保存策略 |
| optimizer | 参数更新和状态访问 | 检查优化器循环与状态 |



In [ ]:
def top_hotspots(events, topk=2):
    """按阶段总耗时排序，并补充其在总时间中的占比。

    events 中每项需要包含 name、total_ms 和 calls；这里处理的是
    已经整理好的事件记录，不负责从 profiler 采集原始事件。
    """
    if not isinstance(events, list) or not events or topk <= 0:
        return []
    required = {'name', 'total_ms', 'calls'}
    if any(not required.issubset(event) for event in events):
        raise ValueError('每个事件都必须包含 name、total_ms 和 calls')
    if any(event['total_ms'] < 0 or event['calls'] <= 0 for event in events):
        raise ValueError('total_ms 不能为负数，calls 必须大于 0')
    total_ms = sum(event['total_ms'] for event in events)
    if total_ms <= 0:
        raise ValueError('事件总耗时必须大于 0')
    ordered = sorted(events, key=lambda event: event['total_ms'], reverse=True)
    return [
        {**event, 'share_pct': round(event['total_ms'] / total_ms * 100, 1)}
        for event in ordered[:topk]
    ]


events = [
    {'name': 'forward', 'total_ms': 38, 'calls': 2},
    {'name': 'backward', 'total_ms': 74, 'calls': 2},
    {'name': 'optimizer', 'total_ms': 18, 'calls': 1},
    {'name': 'dataloader', 'total_ms': 52, 'calls': 4},
]
hotspots = top_hotspots(events)
print('hotspots:', hotspots)
print('focus:', hotspots[0]['name'])
# 输出示例: hotspots -> [('backward', 74), ('dataloader', 52)], focus -> backward


## Q2 验证：热点排序与阶段占比是否正确？

In [ ]:
assert [item['name'] for item in hotspots] == ['backward', 'dataloader']
assert hotspots[0]['share_pct'] == round(74 / 182 * 100, 1)
assert top_hotspots([], topk=2) == []
print('✅ 阶段热点排序与占比计算通过')

## Q3：如何把时间、容量与正确性证据合成一个可验证假设？

17 提供时间热点，18 提供候选峰值、运行时内存差值与容量风险，19 提供结构、数值和梯度是否可信。正确性失败时应先停止性能解释；结果可信后，再根据事先声明的告警阈值判断当前证据是否足以形成实验假设。这里选择的是待验证信号，不直接宣布计算、带宽或 I/O 根因。

| 证据来源 | 输入字段 | 汇总后的作用 |
|:---|:---|:---|
| 时间证据 | hotspot、stage share | 判断哪个阶段值得展开 |
| 容量证据 | dominant object、peak / capacity | 判断是否接近容量上限 |
| 正确性证据 | contract、finite、gradient | 决定性能结果能否继续解释 |


In [ ]:
def synthesize_evidence(
    hotspot, memory_signal, correctness,
    capacity_alert_ratio=0.90, hotspot_alert_share_pct=40.0,
):
    """汇总三类既有证据；不在本页重新计算 profiler 或显存账本。"""
    required_hotspot = {'name', 'share_pct'}
    required_memory = {'dominant_object', 'peak_memory_mb', 'capacity_mb'}
    required_correctness = {'contract_ok', 'finite', 'gradient_ok'}
    if not required_hotspot <= set(hotspot):
        raise KeyError('hotspot 缺少 name 或 share_pct')
    if not required_memory <= set(memory_signal):
        raise KeyError('memory_signal 缺少容量证据字段')
    if not required_correctness <= set(correctness):
        raise KeyError('correctness 缺少正确性证据字段')
    if memory_signal['capacity_mb'] <= 0 or memory_signal['peak_memory_mb'] < 0:
        raise ValueError('capacity_mb 必须为正，peak_memory_mb 不能为负')
    if not 0 <= hotspot['share_pct'] <= 100:
        raise ValueError('share_pct 必须位于 0 到 100 之间')
    if not 0 < capacity_alert_ratio <= 1 or not 0 <= hotspot_alert_share_pct <= 100:
        raise ValueError('告警阈值超出允许范围')

    failed_checks = [name for name, ok in correctness.items() if not ok]
    capacity_ratio = memory_signal['peak_memory_mb'] / memory_signal['capacity_mb']
    if failed_checks:
        status = 'blocked_by_correctness'
        primary_signal = 'correctness'
        hypothesis = '先修复正确性问题，再解释性能结果'
    elif capacity_ratio >= capacity_alert_ratio:
        status = 'ready_for_experiment'
        primary_signal = 'memory'
        hypothesis = f"{memory_signal['dominant_object']} 导致容量压力"
    elif hotspot['share_pct'] >= hotspot_alert_share_pct:
        status = 'ready_for_experiment'
        primary_signal = 'time'
        hypothesis = f"{hotspot['name']} 位于时间关键路径"
    else:
        status = 'need_more_evidence'
        primary_signal = 'none'
        hypothesis = '当前没有单一强信号'

    return {
        'status': status,
        'primary_signal': primary_signal,
        'hypothesis': hypothesis,
        'hotspot': hotspot['name'],
        'hotspot_share_pct': hotspot['share_pct'],
        'dominant_object': memory_signal['dominant_object'],
        'capacity_ratio': round(capacity_ratio, 3),
        'failed_checks': failed_checks,
        'thresholds': {
            'capacity_alert_ratio': capacity_alert_ratio,
            'hotspot_alert_share_pct': hotspot_alert_share_pct,
        },
    }


correctness_ok = {'contract_ok': True, 'finite': True, 'gradient_ok': True}
capacity_evidence = synthesize_evidence(
    hotspot={'name': 'backward', 'share_pct': 40.7},
    memory_signal={'dominant_object': 'activation', 'peak_memory_mb': 61, 'capacity_mb': 64},
    correctness=correctness_ok,
)
input_evidence = synthesize_evidence(
    hotspot={'name': 'dataloader', 'share_pct': 55.0},
    memory_signal={'dominant_object': 'activation', 'peak_memory_mb': 24, 'capacity_mb': 64},
    correctness=correctness_ok,
)
invalid_evidence = synthesize_evidence(
    hotspot={'name': 'forward', 'share_pct': 62.0},
    memory_signal={'dominant_object': 'temporary', 'peak_memory_mb': 30, 'capacity_mb': 64},
    correctness={'contract_ok': True, 'finite': False, 'gradient_ok': True},
)
print('capacity:', capacity_evidence)
print('input:', input_evidence)
print('invalid:', invalid_evidence)


## Q3 验证：正确性门槛、容量压力与时间热点能否正确分流？

In [ ]:
assert capacity_evidence['primary_signal'] == 'memory'
assert capacity_evidence['capacity_ratio'] == round(61 / 64, 3)
assert input_evidence['primary_signal'] == 'time'
assert input_evidence['hotspot'] == 'dataloader'
assert invalid_evidence['status'] == 'blocked_by_correctness'
assert invalid_evidence['failed_checks'] == ['finite']
assert capacity_evidence['thresholds']['capacity_alert_ratio'] == 0.90
print('✅ 三类证据门槛与待验证信号分流通过')

## Q4：如何把证据假设变成可判断的单变量实验？

每次实验先固定模型、输入、batch、dtype、设备、重复次数和测量方法，再由学习者显式提交一个候选变化。运行后分别保存基线与候选的重复样本，单独记录失败运行，再比较中位数和 P95。候选变化必须同时超过最小收益阈值和基线自身波动，并通过质量门槛；本页不替具体专题选择 checkpoint、量化或并行策略。示例用 `(P95 - median) / median` 作为教学用波动代理，它不是统计置信区间，真实项目需要更多样本和独立回归协议。

| 实验字段 | 需要记录的内容 |
|:---|:---|
| hypothesis | 哪个信号值得用实验验证 |
| fixed conditions | model、input、batch、dtype、device、repeats |
| single change | 基线与候选唯一不同的变量 |
| repeated samples | 基线与候选的成功样本和失败运行 |
| metrics | 中位数、P95、质量与资源代价 |
| regression threshold | 最小收益与基线波动两者的较大值 |
| decision | accept、reject 或 rerun，并保留原因 |


In [ ]:
def design_single_variable_experiment(evidence, baseline, candidate_change=None):
    """固定 baseline，并校验 candidate_change 只改变一个变量。"""
    required = {'model', 'input_shape', 'batch_size', 'dtype', 'device', 'repeats'}
    missing = sorted(required - set(baseline))
    if missing:
        raise KeyError(f'baseline 缺少固定条件: {missing}')
    fixed_conditions = {key: baseline[key] for key in sorted(required)}

    if evidence['status'] == 'blocked_by_correctness':
        return {
            'action': 'fix_correctness_first',
            'hypothesis': evidence['hypothesis'],
            'baseline': dict(baseline),
            'candidate': None,
            'changed_variable': None,
            'fixed_conditions': fixed_conditions,
            'metrics': ['contract_ok', 'finite', 'gradient_ok'],
            'evidence_level': 'teaching_proxy',
        }

    if evidence['status'] != 'ready_for_experiment' or not candidate_change:
        return {
            'action': 'collect_more_evidence',
            'hypothesis': evidence['hypothesis'],
            'baseline': dict(baseline),
            'candidate': None,
            'changed_variable': None,
            'fixed_conditions': fixed_conditions,
            'metrics': ['latency_ms', 'throughput_tok_s'],
            'evidence_level': 'teaching_proxy',
        }

    if len(candidate_change) != 1:
        raise ValueError('candidate_change 必须且只能包含一个变量')
    changed_variable, changed_value = next(iter(candidate_change.items()))
    if baseline.get(changed_variable) == changed_value:
        raise ValueError('候选变量必须真正改变')
    candidate = dict(baseline)
    candidate[changed_variable] = changed_value
    return {
        'action': 'run_comparison',
        'hypothesis': evidence['hypothesis'],
        'baseline': dict(baseline),
        'candidate': candidate,
        'changed_variable': changed_variable,
        'fixed_conditions': fixed_conditions,
        'metrics': ['latency_ms', 'throughput_tok_s', 'quality', 'resource_cost'],
        'evidence_level': 'teaching_proxy',
    }


baseline = {
    'model': 'TinyLM',
    'input_shape': (4, 128),
    'batch_size': 4,
    'dtype': 'fp32',
    'device': 'cpu',
    'repeats': 5,
    'checkpoint': False,
    'num_workers': 0,
}
memory_plan = design_single_variable_experiment(
    capacity_evidence, baseline, {'checkpoint': True}
)
input_plan = design_single_variable_experiment(
    input_evidence, baseline, {'num_workers': 2}
)
blocked_plan = design_single_variable_experiment(invalid_evidence, baseline)
print('memory plan:', memory_plan)
print('input plan:', input_plan)
print('blocked plan:', blocked_plan)


## Q4 验证：单变量约束、失败运行与回归阈值是否同时生效？

In [ ]:
def changed_keys(plan):
    """返回 baseline 与 candidate 之间真正变化的配置项。"""
    if plan['candidate'] is None:
        return []
    keys = set(plan['baseline']) | set(plan['candidate'])
    return sorted(key for key in keys if plan['baseline'].get(key) != plan['candidate'].get(key))


def percentile(values, fraction):
    """使用最近秩方法计算教学用分位数。"""
    if not values or not 0 <= fraction <= 1:
        raise ValueError('values 不能为空，fraction 必须位于 [0, 1]')
    ordered = sorted(values)
    index = max(0, min(len(ordered) - 1, int((len(ordered) * fraction + 0.999999) - 1)))
    return ordered[index]


def summarize_repeated_runs(runs):
    """分开记录成功样本与失败运行，并汇总延迟和质量。"""
    successful = [run for run in runs if run.get('ok') is True]
    failed = [run for run in runs if run.get('ok') is not True]
    if any('latency_ms' not in run or 'quality' not in run for run in successful):
        raise KeyError('成功运行必须包含 latency_ms 和 quality')
    latencies = [run['latency_ms'] for run in successful]
    qualities = [run['quality'] for run in successful]
    if any(value <= 0 for value in latencies):
        raise ValueError('成功运行的 latency_ms 必须大于 0')
    if not successful:
        return {'valid_runs': 0, 'failed_runs': len(failed), 'median_ms': None, 'p95_ms': None, 'quality': None}
    return {
        'valid_runs': len(successful),
        'failed_runs': len(failed),
        'median_ms': percentile(latencies, 0.5),
        'p95_ms': percentile(latencies, 0.95),
        'quality': sum(qualities) / len(qualities),
    }


def evaluate_regression(baseline_runs, candidate_runs, min_valid_runs=5, min_improvement=0.05, quality_tolerance=0.01):
    """判断候选收益是否超过 baseline 波动并通过质量门槛。"""
    baseline_summary = summarize_repeated_runs(baseline_runs)
    candidate_summary = summarize_repeated_runs(candidate_runs)
    if min(baseline_summary['valid_runs'], candidate_summary['valid_runs']) < min_valid_runs:
        return {'decision': 'rerun', 'reason': 'insufficient_valid_runs', 'baseline': baseline_summary, 'candidate': candidate_summary}
    baseline_noise = (baseline_summary['p95_ms'] - baseline_summary['median_ms']) / baseline_summary['median_ms']
    required_improvement = max(min_improvement, baseline_noise)
    observed_improvement = (baseline_summary['median_ms'] - candidate_summary['median_ms']) / baseline_summary['median_ms']
    quality_ok = candidate_summary['quality'] >= baseline_summary['quality'] - quality_tolerance
    decision = 'accept' if observed_improvement >= required_improvement and quality_ok else 'reject'
    return {
        'decision': decision,
        'reason': 'threshold_and_quality_checked',
        'baseline': baseline_summary,
        'candidate': candidate_summary,
        'baseline_noise': baseline_noise,
        'required_improvement': required_improvement,
        'observed_improvement': observed_improvement,
        'quality_ok': quality_ok,
    }


baseline_runs = [
    {'ok': True, 'latency_ms': value, 'quality': 0.900}
    for value in (100, 102, 99, 101, 100)
]
candidate_runs = [
    {'ok': True, 'latency_ms': value, 'quality': 0.895}
    for value in (88, 90, 89, 91, 87)
] + [{'ok': False, 'failure': 'timeout'}]
regression_result = evaluate_regression(baseline_runs, candidate_runs)
print('regression_result:', regression_result)


assert memory_plan['action'] == 'run_comparison'
assert changed_keys(memory_plan) == ['checkpoint']
assert input_plan['action'] == 'run_comparison'
assert changed_keys(input_plan) == ['num_workers']
assert input_plan['candidate']['num_workers'] == baseline['num_workers'] + 2
assert input_plan['fixed_conditions']['input_shape'] == baseline['input_shape']
assert blocked_plan['action'] == 'fix_correctness_first'
assert blocked_plan['candidate'] is None
assert all(plan['evidence_level'] == 'teaching_proxy' for plan in (memory_plan, input_plan, blocked_plan))
assert regression_result['candidate']['failed_runs'] == 1
assert regression_result['observed_improvement'] > regression_result['required_improvement']
assert regression_result['quality_ok'] is True
assert regression_result['decision'] == 'accept'
assert evaluate_regression(baseline_runs[:2], candidate_runs[:2])['decision'] == 'rerun'
print('✅ 单变量约束、失败运行与回归阈值通过')

## 相关阅读
**导语：** 完成本节后，可以进入性能归因页或专题路线，将这里的实验协议用于真实证据。
- [Part 01 · 13 性能分析与瓶颈归因](../01_Hardware_Math_and_Systems/13_Profiling_and_Bottleneck_Analysis.ipynb)
- [性能分析专题](../topic_discussion/performance_analysis/intro.md)
- [PyTorch Profiler 官方教程](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)

